# 🏋️ LoRA 训练实战 — 从数据准备到模型合并

**本文目标**：掌握 LoRA 微调的完整工作流——数据准备、超参数调参、训练监控、模型合并与推送。

读完这篇你会理解：
- 数据准备的最佳实践 (格式、长度分布、配比)
- rank/alpha/dropout 的调参策略
- 训练监控指标 (loss、grad norm、eval PPL)
- Merge vs Unmerge 的取舍
- 常见踩坑与解决方案

## 1. 数据准备

### 1.1 数据量经验法则

```
任务复杂度    推荐数据量      LoRA rank
简单指令微调  500-2000 条    rank=8-16
领域适配      1000-5000 条   rank=16-32
复杂任务      2000-10000 条  rank=32-64
长上下文扩展  500-2000 条    rank=8-16 (配合 YaRN)
代码/数学     5000-20000 条  rank=32-64
```

### 1.2 数据格式

```python
# 标准 Chat Format
data = [
    {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "解释量子计算"},
            {"role": "assistant", "content": "量子计算利用量子比特..."}
        ]
    },
    # ... more examples
]

# 训练时: 拼接为 token 序列, 只在 assistant 部分计算 loss
# [SYSTEM] [USER] [ASSISTANT]
#                    ↑ 只在这部分回传梯度
```

### 1.3 长度分布策略

```python
# 正确做法: 混合不同长度
length_distribution = {
    512:  0.3,    # 30% 短样本 (快速收敛基础模式)
    1024: 0.3,    # 30% 中样本
    2048: 0.2,    # 20% 长样本
    4096: 0.15,   # 15% 特长样本
    8192: 0.05,   # 5% 超长 (长上下文扩展的关键)
}

# 错误做法: 全部 8K+ 长文本
# → 训练慢, 且容易让模型"忘记"短上下文能力
```

## 2. 超参数调参

### 2.1 Rank 和 Alpha

```
rank 的选择:
  rank=4:  "快速实验"级别, 效果已经不错 (说明微调确实低秩)
  rank=8:  "轻量适配"级别, 简单任务和位置扩展
  rank=16: "通用微调"级别, 默认推荐
  rank=32: "复杂任务"级别, 代码/数学/多语言
  rank=64: "逼近全量微调"级别, 边际收益递减

alpha 的选择:
  经验法则: alpha = rank (保持 scale=1)
  或 alpha = 2 × rank (更激进的 LoRA 修正)

关系:
  effective_step_size = lr × (alpha / rank)
  → alpha/rank 实际上是一个"学习率缩放因子"
  → 可以用 lr 补偿 rank 的变化
```

### 2.2 学习率调度

```python
# LoRA 的推荐学习率 (比全量微调高 10-100x)
learning_rates = {
    "rank=4":  2e-4,
    "rank=8":  2e-4,
    "rank=16": 1e-4,
    "rank=32": 5e-5,
    "rank=64": 2e-5,
}

# 调度器
lr_scheduler = "cosine"  # 推荐
warmup_ratio = 0.03      # 3% 的 step 做 warmup

# 为什么 LoRA 可以用更激进的学习率?
# → 预训练权重 frozen, 不会"灾难性遗忘"
# → A 和 B 初始化为小值, 需要较大的 lr 才能有效更新
```

### 2.3 其他关键参数

```
dropout: 0.05-0.1  (防止过拟合, 小数据集尤其重要)
batch_size: 4-16 (取决于 GPU 显存)
gradient_accumulation: 使 effective batch = 32-128
epochs: 1-3 (LoRA 收敛快, 不需要多 epoch)
  → 如果数据 < 1000 条, 可以用 3-5 epoch
  → 如果数据 > 5000 条, 1 epoch 通常足够
```

## 3. Merge vs Unmerge

### 3.1 两种使用方式

```
方式 A: Unmerged (Adapter 模式)
  保留预训练权重 + LoRA adapter 分离
  W = W_pretrained (frozen)
  推理时: output = W@x + B@A@x
  → 两次矩阵乘法, 略慢 ~3-5%
  → 但可以热切换 LoRA adapter

方式 B: Merged
  把 LoRA 权重"合并"回预训练权重
  W_merged = W_pretrained + B@A
  推理时: output = W_merged @ x
  → 一次矩阵乘法, 与原始模型同样快
  → 但失去灵活性 (不能热切换)

什么时候 Merge?
  - 部署到生产 (追求推理速度)
  - llama.cpp 推理 (不支持 unmerged LoRA)
  - 模型发布 (一个合并后的完整权重)

什么时候不 Merge?
  - 多 LoRA 服务 (vLLM multi-LoRA)
  - 实验阶段 (快速迭代)
  - 需要频繁切换 adapter
```

### 3.2 Merge 代码

```python
# 标准 merge
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")
lora_model = PeftModel.from_pretrained(base_model, "./lora-checkpoint")

# Merge
merged_model = lora_model.merge_and_unload()

# 保存
merged_model.save_pretrained("./llama-2-7b-lora-merged")
```

## 4. 常见踩坑

```
坑 1: "为什么我的 LoRA 没有任何效果？"
  → 检查: alpha / rank 的缩放是否正确
  → 检查: 是否不小心 freeze 了 A 或 B
  → 检查: 学习率是否太小 (LoRA 需要比全量微调大 10-100x)

坑 2: "训练 loss 下降但 eval 很差"
  → 过拟合: 增加 dropout、减少 epoch、增加数据
  → rank 太大: 小数据集用 rank=4-8

坑 3: "Merge 后模型坏了"
  → 检查: alpha/rank 是否在 merge 时正确应用
  → 检查: 是否用了正确的 dtype (BF16 vs FP16)

坑 4: "继续训练 checkpoint"
  → LoRA adapter 保存的是 A 和 B, 不是 ΔW
  → 恢复时: adapter = PeftModel.from_pretrained(base, ckpt)
  → 不需要保存 base model (除非 merge)

坑 5: "训练后短上下文能力下降"
  → 训练数据全部是长文本 → 模型"忘记"短上下文
  → 解法: 混合长度分布 (见 §1.3)
```

In [ ]:
# LoRA 训练的规模估算

def estimate_lora_training(model_params, rank, target_layers, data_samples,
                            seq_len, batch_size, gpu_mem_gb, fp="bf16"):
    """估算 LoRA 训练的资源需求"""

    bytes_per_param = 2 if fp == "bf16" else 4

    # LoRA 参数量
    d_model = int(model_params ** 0.5 * 100)  # 粗略估算
    n_layers = model_params // (12 * d_model * d_model)
    lora_per_layer = 2 * rank * d_model  # A + B
    lora_params = lora_per_layer * n_layers * target_layers
    lora_mem = lora_params * bytes_per_param  # LoRA 权重

    # 优化器状态
    optimizer_mem = lora_mem * 3  # Adam: param + m + v

    # 基础模型 (QLoRA 4-bit)
    base_model_gb = model_params * 1e9 * 1.5 / (1024**3) if "q" in fp else                     model_params * 1e9 * bytes_per_param / (1024**3)

    # 激活值 (粗略)
    activation_gb = batch_size * seq_len * d_model * n_layers * bytes_per_param / (1024**3) * 3

    total_gb = base_model_gb + lora_mem / (1024**3) + optimizer_mem / (1024**3) + activation_gb

    # 训练时间估算
    tokens_per_step = batch_size * seq_len
    total_tokens = data_samples * seq_len
    total_steps = total_tokens / tokens_per_step
    time_per_step = 0.5  # 秒 (粗略)
    total_hours = total_steps * time_per_step / 3600

    return total_gb, total_hours

print("LoRA/QLoRA 训练资源估算")
print("=" * 60)

for name, params, rank, layers, fp in [
    ("LLaMA-7B LoRA", 7, 16, 4, "bf16"),
    ("LLaMA-7B QLoRA", 7, 16, 4, "q4_bf16"),
    ("LLaMA-13B QLoRA", 13, 16, 4, "q4_bf16"),
    ("LLaMA-70B QLoRA", 70, 16, 4, "q4_bf16"),
]:
    mem, hrs = estimate_lora_training(params, rank, layers, 1000, 2048, 4, 24, fp)
    feasible = "OK" if mem < 24 else "NEED " + str(int(mem/24 + 1)) + "x GPU"
    print(f"  {name:20s}: {mem:.1f} GB, ~{hrs:.1f}h ({feasible})")